In [1]:
import openai
import json
# from utils import Spotify
import tiktoken
import time

In [2]:
import os
# os.urandom(24)

from os.path import join, dirname
join(dirname('utils.py'), '.env')

'.env'

In [18]:
def remove_cache_file(directory="."):
    """
    Removes the .cache file in the specified directory if it exists.
    
    Parameters:
        directory (str): The directory to look for the .cache file. Defaults to the current directory.
        
    Returns:
        bool: True if the file was removed, False if the file did not exist.
    """
    cache_path = os.path.join(directory, ".cache")
    
    if os.path.exists(cache_path):
        os.remove(cache_path)
        print(f"Removed: {cache_path}")
    else:
        print(f"No .cache file found in {directory}.")
    
remove_cache_file()

Removed: ./.cache


In [4]:
with open('../instance/config.json') as config_file:
    
    config = json.load(config_file)

In [5]:
from spotipy.oauth2 import SpotifyClientCredentials,SpotifyOAuth
# from spotipy.oauth2 import 
import spotipy 
import pandas as pd
FEATURES = [
    'danceability', 'energy', 'acousticness', 'instrumentalness', 'valence', 'loudness', 'tempo',
]
class Spotify:
    """
    ---------------------------------------------------------------------------------------------
    Spotify class helps to extract tracks and its audio features from user playlists
    ---------------------------------------------------------------------------------------------
    Parameters:
        - client_id (str): User client id
        - client_secret (str): User secret id
    ---------------------------------------------------------------------------------------------
    Attributes:
        - client_ (spotipy.Spotify object): Initialized Spotify client object  
        - playlists_name_ (List[str]): List of all playlist names
        - df_ (pandas.DataFrame): Data with tracks and audio features
    ---------------------------------------------------------------------------------------------
    Methods
        - get_tracks_from_playlists: Extract tracks and audio features from user playlists
    ---------------------------------------------------------------------------------------------
    """
    def __init__(self, client_id=None, client_secret=None, redirect_uri=None, auth_token=None, scope = "playlist-read-private playlist-read-collaborative playlist-modify-public user-library-read" ):
        self.client_id=client_id
        self.client_secret=client_secret
        self.redirect_uri = redirect_uri
        self.scope = scope
        self.auth_token=auth_token
    
    def connect(self):
        # client_creds = SpotifyClientCredentials(client_id=self.client_id, 
        #                                         client_secret=self.client_secret)
        
        # client = spotipy.Spotify(client_credentials_manager=client_creds)
        # client_creds = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret, redirect_uri=SPOTIFY_REDIRECT_URI)
        if self.auth_token is not None:
            client =  spotipy.Spotify(auth=self.auth_token)
            
        else:
            client = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=self.client_id,
                                                client_secret=self.client_secret,
                                                redirect_uri=self.redirect_uri,
                                                scope=self.scope))
        self.client_ = client


        
    def get_playlists_from_user(self) -> pd.DataFrame:
        playlists = []
        offset = 0
        while True:
            response = self.client_.current_user_playlists(offset=offset, limit=50)
            if response is not None:
                playlists.extend(response['items'])
            if response['next']:
                offset += len(response['items'])
            else:
                break
        # print(playlists)
        playlists = [playlist for playlist in playlists if playlist is not None]
        self.playlists_detail = playlists
        self.playlists_name_ = [playlist['name'] for playlist in playlists]
        return pd.DataFrame(self.playlists_detail)
    
    def get_tracks_from_playlists(self, playlists=None):
        """
        ------------------------------------------------------------------------------------------
        get_tracks_from_playlists connects to spotify api and extracts all tracks and 
                                  audio features from playlists
        ------------------------------------------------------------------------------------------
        Parameters:
            - playlists (List[str]): List of playlist names
            - unique (Bool): Returns unique tracks if true
        ------------------------------------------------------------------------------------------
        Returns:
            - Pandas DataFrame of with all tracks in 
        ------------------------------------------------------------------------------------------
        """
        track_ls = []
        playlists_ls = self.playlists_name_ if playlists is None else playlists
        
        for playlist in self.playlists_detail:
            name = playlist['name']
            is_public = 1 if playlist['public'] else 0  # Check if the playlist is public
            
            if name not in playlists_ls:
                continue
            
            results = self.client_.playlist(playlist['id'], fields="tracks,next")
            tracks = results['tracks']

            for i, item in enumerate(tracks['items']):
                artists = ', '.join(artist['name'] for artist in item['track']['artists'])
                track_ls.append((name, item['track']['id'], item['track']['name'], artists, is_public))  # Include artists
        
        self.tracks_df_ = pd.DataFrame(track_ls, columns=['playlist_name', 'track_id', 'track_name', 'artist_names', 'is_public']).drop_duplicates()
        return self.tracks_df_

    def get_playlists_df(self) -> pd.DataFrame:
        playlist_df = pd.DataFrame(self.playlists_detail)
        return playlist_df
    
    def get_tracks_df(self) -> pd.DataFrame:
        # cols = ['name'] + FEATURES
        tracks_df = self.tracks_df_[['track_id', 'track_name', 'artist_names']].drop_duplicates()
        return tracks_df
    
    def create_playlist(self, playlist_name, song_list):
        # Step 1: Create a public playlist
        user_id = self.client_.current_user()["id"]
        playlist = self.client_.user_playlist_create(user=user_id, name=playlist_name, public=True)
        playlist_id = playlist['id']

        # Step 2: Search for each song and add to the playlist
        # track_ids = []
        # for song in song_list:
        #     results = self.client_.search(q=song, type='track', limit=1)
        #     if results['tracks']['items']:
        #         track_ids.append(results['tracks']['items'][0]['id'])
        tracks_df = self.get_tracks_df()
        tracks_df = tracks_df.groupby(['track_name','artist_names'],as_index=False).first()
        track_ids = tracks_df[tracks_df['track_name'].isin(song_list)].track_id.tolist()
        if track_ids:
            self.client_.playlist_add_items(playlist_id, track_ids)
            print(f"Playlist '{playlist_name}' created with {len(track_ids)} songs!")
        else:
            print("No valid tracks found to add to the playlist.")


In [7]:
# Obtain user credentials
client_id = config['client_id_main']
client_secret = config['client_secret_main']
redirect_uri = config['redirect_uri']

# Initialize
sp = Spotify(client_id, client_secret, redirect_uri)

# Connect
sp.connect()

# Get list of playlist and allow user to select which playlists to extract songs from
playlist_df = sp.get_playlists_from_user()


# # # Get all tracks from the selected playlist
tracks_df = sp.get_tracks_from_playlists()


In [21]:
tracks_df

,playlist_name,track_id,track_name,artist_names,is_public
0,Rap 3,4KW1lqgSr8TKrvBII0Brf8,Father Stretch My Hands Pt. 1,Kanye West,1
1,Rap 3,3eekarcy7kvN4yt5ZFzltW,HIGHEST IN THE ROOM,Travis Scott,1
2,Rap 3,27a1mYSG5tYg7dmEjWBcmL,CAN'T SAY,Travis Scott,1
3,Rap 3,6zFMeegAMYQo0mt8rXtrli,HOLIDAY,Lil Nas X,1
4,Rap 3,6gi6y1xwmVszDWkUqab1qw,OUT WEST (feat. Young Thug),"JACKBOYS, Travis Scott, Young Thug",1
...,...,...,...,...,...
758,Old,0BRvjLhMa3Qie8LMnRtY4f,I'm Beginning To See The Light,The Ink Spots,1
759,Old,4SKt36xFNge5y6yeR8n80v,It's Funny To Everyone But Me,The Ink Spots,1
760,Old,1irsnS9Hy4f5UHszpwaKno,When The Swallows Come Back To Capistrano,The Ink Spots,1
761,Old,1owGm6QDclYTJ40SMz97rW,No Moon At All - Remastered,Julie London,1


# Gemini

In [9]:
def df_shuffle(df, max_records=1000, random_state=2024):
    n = min(df.shape[0], max_records)
    return df.sample(n=n, random_state=random_state)


In [19]:
# %pip install -U -q "google-generativeai>=0.8.3"

# ToDo

Note: you may need to restart the kernel to use updated packages.


## Todo
- Fix Creativity parameter

In [19]:
import google.generativeai as genai
import time
import re
# from IPython.display import HTML, Markdown, display


In [20]:
GOOGLE_API_KEY = config['google_api_key']

class GeminiPlaylistCurator:
  def __init__(self, api_key, model_name='gemini-1.5-flash'):
    self.api_key = api_key
    self.model_name = model_name

  def init_chat(self):
    genai.configure(api_key=self.api_key)
    self.bot = genai.GenerativeModel(self.model_name)
    self.chat = self.bot.start_chat(history=[])

  def get_example_df(self):
    example_df_path = 'Example_df_input.csv'
    return pd.read_csv(example_df_path)

  def shuffle_df(self, df, max_rows=10, random_state=2024):
    n = min(df.shape[0], max_rows)
    return df.sample(n=n, random_state=random_state)

  def check_params(self, playlist_name, creativity, min_tracks, max_tracks, special_request):
    if not isinstance(playlist_name, str) or not playlist_name.strip():
      raise ValueError(f"playlist_name must be a non-empty string, value: {playlist_name}")
    if not isinstance(creativity, int) or (creativity < 1) or (creativity > 10):
      raise ValueError(f"creativity must be an integer between 1 and 10, value: {creativity}")
    if not isinstance(min_tracks, int) or (min_tracks < 1):
      raise ValueError(f"min_tracks must be an integer greater than 1, value: {min_tracks}")
    if not isinstance(max_tracks, int) or (max_tracks < 1):
      raise ValueError(f"max_tracks must be an integer greater than 1, value: {max_tracks}")
    if not isinstance(special_request, str):
      raise ValueError(f"special_request must be a string, value: {special_request}")


  def get_prompt(self, df, playlist_name, creativity, min_tracks, max_tracks, special_request):
    prompt = f"""
        You are a spotify music playlist curator who organizes tracks into thematic playlists based on the given list of tracks. 
        The inputs will be given in the following format:
          - playlist name: <str>
          - creativity: <int>
          - special requests: <text>
          - data: 
          ```
          <text>
          ```

        Input format:
        - Playlist name: Title of the playlist, guiding the theme.
        - creativity: Scale of creativity (1 to 10).
        - Special requests: Additional selection criteria.
        - Data: CSV format with track_name and artist_name. 

        Notes:
        - The playlist name will be the title of the playlist. Based on the playlist name, extract tracks in 'data' that you feel is most related to
          the theme and respond with the list of tracks belonging in the playlist. You should consider the songs audio features such as genre, danceability, 
          energy, acousticness, instrumentalness, valence, loudness, tempo and other where you find are helpful.
        - The creativity is how creative you can be when curating the playlist. It ranges from 1 to 10. At creativity 1, strictly match tracks by the most 
          relevant audio features and themes in the playlist name. At creativity 10, allow for more diverse but still loosely related tracks.
        - Special requests: this is some additional information that will help you choose which songs to pick in the playlist.
        - The data will contain track information and is the collection of tracks to be extracted from. It will be in csv format with the 
          following columns:
            - track_name: Name of track
            - artist_name: Name of artists separated by commas
        
        Refer to the input below:
        playlist name: {playlist_name}
        creativity: {creativity}
        special request: {special_request}
        data: 
        ```
        {df.to_csv(index=False, na_rep='NA')}
        ```

        Restrictions:
        - Only extract tracks from the input data.
        - Return between {min_tracks} and {max_tracks} tracks. If none match the playlist theme, return an empty string.
        - Format: songA, songB, songC,... (comma-separated list without additional text).
        
        Before sending back the response, re-check how many tracks are there and select maximum {max_tracks} tracks.
        """
    return prompt
  
  def get_chatgpt_generated_prompt(self, df, playlist_name, creativity, min_tracks, max_tracks, special_request):
    prompt = f"""
      You are a Spotify music playlist curator who creates thematic playlists based on the given list of tracks based on a playlist name. 
      ### Guidelines for Curation
      - Select tracks most relevant to the playlist name and theme.
      - Use audio features like genre, danceability, energy, acousticness, valence, loudness, tempo, and instrumentalness where appropriate.
        - Example: For "Relaxing Evening," choose tracks with low energy, high acousticness, and low tempo.
        - Example: For "Party," prioritize high danceability, energy, and tempo.
      - Incorporate special requests when provided.
      ### Input Format
      - **Playlist name:** Title of the playlist theme.
      - **creativity:** Scale of creativity (1 to 10).
        - At 1, strictly match tracks by audio features and theme.
        - At 10, allow for broader, creative selections still loosely related to the theme.
      - **Special requests:** Additional guidance for track selection (e.g., "focus on upbeat songs").
      - **Data:** A CSV table of eligible tracks enclosed with triple backticks with these columns:
        - `track_name`: Name of track.
        - `artist_name`: Names of artists (comma-separated).
      ### Output Format
      - Return a comma-separated list of tracks closed with double quotes, example: "songA","songB","songC".
      - Restrictions:
        - Only use tracks from the input data.
        - Extract `{min_tracks}-{max_tracks}` tracks.
        - If no tracks match, return an empty string.
      
      ### Example:
      #### Example input:
      - **Playlist name:** Only rap
      - **creativity:** 1
      - **Special request:** I only want rap songs
      - **Data:**
      ```
      {self.get_example_df().to_csv(index=False, na_rep='NA')}
      ```

      #### Example ouput:
      "Father Stretch My Hands Pt. 1", "HIGHEST IN THE ROOM", "CAN'T SAY", "HOLIDAY"

      ### Inputs:
      - **Playlist name:** {playlist_name}
      - **creativity:** {creativity}
      - **Special request:** {special_request}
      - **Data:**
      ```
      {df.to_csv(index=False, na_rep='NA')}
      ```
      Strictly follow the output format above and before sending back the response, count how many tracks are generated and select maximum {max_tracks} tracks.
      """
    return prompt


  def ask_gemini(self, df, playlist_name, creativity, min_tracks, max_tracks, special_request=None, max_rows=10, random_state=2024):
    # prompt = self.get_prompt(df, playlist_name, creativity, min_tracks, max_tracks, special_request)
    df = self.shuffle_df(df,max_rows,random_state)
    self.check_params(playlist_name, creativity, min_tracks, max_tracks, special_request)
    prompt = self.get_chatgpt_generated_prompt(df, playlist_name, creativity, min_tracks, max_tracks, special_request)
    response = self.chat.send_message(prompt)
    print(f"{response.text[:50]}...{response.text[-30:]}")
    # songs = [song.strip() for song in response.text.split("\t")]
    songs = re.findall(r'"([^"]*)"', response.text)
    return songs, response



# df = sp.get_df().drop_duplicates()
df = sp.get_tracks_df()[['track_name', 'artist_names']].drop_duplicates()
playlist_name = "I'm hella stressed"
creativity =  10
min_tracks =  10
max_tracks = 30
special_request = "I want loud music"

# gemini_obj = GeminiPlaylistCurator(GOOGLE_API_KEY)
# gemini_obj.init_chat()
# songs = gemini_obj.ask_gemini(df, playlist_name, creativity, min_tracks, max_tracks, special_request)

# prompt = get_prompt(df, playlist_name, creativity, min_tracks, max_tracks, special_request)
songs_list=[]
response_list=[]
for i in range(3):
  print(f"current iteration: {i+1}")
  gemini_obj = GeminiPlaylistCurator(GOOGLE_API_KEY)
  gemini_obj.init_chat()
  songs, response = gemini_obj.ask_gemini(df, playlist_name, creativity, min_tracks, max_tracks, special_request)
  print(f"Number of tracks: {len(songs)}")
  songs_list.append(songs)
  response_list.append(response.text)
  print("\n")
  time.sleep(3)

current iteration: 1
"Decode","Suede","euphoria"
..."Decode","Suede","euphoria"

Number of tracks: 3


current iteration: 2
"Decode","Suede","euphoria"
..."Decode","Suede","euphoria"

Number of tracks: 3


current iteration: 3
"Decode","Suede","euphoria"
..."Decode","Suede","euphoria"

Number of tracks: 3




In [24]:
print(gemini_obj.get_example_df().to_csv(index=False, na_rep='NA'))

Unnamed: 0,track_name,artist_names
0,Father Stretch My Hands Pt. 1,Kanye West
1,HIGHEST IN THE ROOM,Travis Scott
2,CAN'T SAY,Travis Scott
3,HOLIDAY,Lil Nas X
759,No Moon At All - Remastered,Julie London
760,Maybe,The Ink Spots



In [35]:
print(response_list[0])

```python
import pandas as pd

def create_playlist(playlist_name, temperature, special_requests, data):
    """
    Creates a Spotify playlist based on given parameters.
    """
    df = pd.read_csv(data)
    
    #Simplified selection based on special requests (Loud music).  A more sophisticated approach would analyze audio features.
    selected_tracks = df[~df['track_name'].str.contains("Acoustic")] #remove acoustic tracks

    #Limit to 30 tracks
    selected_tracks = selected_tracks.head(min(30, len(selected_tracks)))

    #Return comma separated list
    return '"' + '","'.join(selected_tracks['track_name'].tolist()) + '"'


playlist_name = "I'm hella stressed"
temperature = 1
special_requests = "I want loud music"
data = """track_name,artist_names
"thank u, next - Acoustic",Bailey Rushlow
Summer of Farewells (From Up On Poppy Hill),kno Piano Music
Easy,Mac Ayres
human,Christina Perri
Bang Bang (My Baby Shot Me Down),Niomí
fujifreestyle,"Blvck Svm, OBEEHAVE"
"Requiem In D Minor, 

# Add to Spotify account

In [13]:
sp.create_playlist(playlist_name, songs)

Playlist 'Rap heavy' created with 30 songs!


['6xcJyGpfZbuuiequtnlKt4',
 '4LaGu95Ui2s4vprSQYWUAZ',
 '27a1mYSG5tYg7dmEjWBcmL',
 '6BU1RZexmvJcBjgagVVt3M',
 '4KW1lqgSr8TKrvBII0Brf8',
 '6pcywuOeGGWeOQzdUyti6k',
 '3eekarcy7kvN4yt5ZFzltW',
 '6zFMeegAMYQo0mt8rXtrli',
 '6wsqVwoiVH2kde4k4KKAFU',
 '2tudvzsrR56uom6smgOcSf',
 '6gi6y1xwmVszDWkUqab1qw',
 '52NGJPcLUzQq5w7uv4e5gf',
 '0WtDGnWL2KrMCk0mI1Gpwz',
 '2FDTHlrBguDzQkp7PVj16Q',
 '76gcXhY3Zv6wW0BTe9nHJo',
 '0KKkJNfGyhkQ5aFogxQAPU',
 '0QIjsbm2fh1cJ45XO9eGqq',
 '3FNy4yzOhHhFBeA5p4ofoq',
 '2FoahzOSxJnalPA8aBUme3',
 '4yreExU3eRNTe2iJz6X6k3',
 '50a8bKqlwDEqeiEknrzkTO',
 '02Cp3VTUWNed8hr69BhKz6',
 '77DRzu7ERs0TX3roZcre7Q',
 '3QFInJAm9eyaho5vBzxInN',
 '5KI7I4mEtulXcv5VQJaV35',
 '1D3z6HTiQsNmZxjl7F7eoG',
 '6x9pCndnXEoea0CMcfjs9W',
 '2yUzr8Sr6ldG8vmHhZwTnz',
 '4yLyVdEqV790aIXyGif85v',
 '52eIcoLUM25zbQupAZYoFh']

# OpenAI

In [15]:
# col_name_dict = {
#     'danceability': 'dance'
#     'acousticness': 'acoustic',
#     'instrumentalness': 'instrum',
#     # 'duration_ms': 'duration',
#     # 'time_signature': 'timesig',
#     # 'speechiness': 'speech',
#     # 'loudness': 'loud',
# }
def count_tokens(messages):
    total_tokens = 0
    for message in messages:
        # Role adds an extra token for each message
        role_token_count = len(encoding.encode(message['role']))
        # Content token count
        content_token_count = len(encoding.encode(message['content']))
        # Add both role and content tokens to total count
        total_tokens += role_token_count + content_token_count + 2  # +2 for separators (message overhead)
    return total_tokens

In [16]:
# Things to do to reduce tokens
## Round decimals. So they might have similar tokens
## Feature selection
## Change feature names. i.e. accoustincess to accoustic
## Batching:
##    - For each iteration, identify tracks that might be a part of the playlist, and aggregate the result

# Things to add
## A creativity (0 - 1) mark to allow users to tell how closely related should a track be to be a part of the playlist.
## For example: If creativity is closer to 1 then more tracks will be added as the model is allowed to be more creative.
##              If creativity is closer to 0, then less tracks will be added as model less creative.
## Distribution or skewness of creativity??

## Add more details on how the songs should be chosen, i.e. imagine you are going through all your songs saved in your spotify playlist to create another playlist titled ...
## Maybe add more descriptive words like feels and mood 
## Maybe try to give a small example.
## Instead of creativity being a continuous value, maybe have it as a discrete value 1,2,3 whre the prompt will change bsaed on the value.
## Maybe have another variable (flexibility), which includes flexibility to include songs from a wider range of genre

In [17]:
# Based on playlist name
from sklearn.utils import shuffle

# def get_prompt_with_music_features():
    
# Load the tokenizer for GPT-4 (gpt-4 and gpt-3.5-turbo use the same encoding)
encoding = tiktoken.encoding_for_model("gpt-4o")

def generate_playlist_from_name(playlist_name, creativity, df):
    openai.api_key = config['openai_api_key']
    system_content = """
    You are a spotify music playlist curator who organizes tracks into thematic playlists based on their audio features extracted
    from spotify api. 
    - The inputs will be given in the following format:
      playlist name: ...
      creativity: ...
      data: ```...```
    - The data in csv format will be supplied and the features below will be given for each track. These features can be helpful 
      in clustering the tracks into playlists but not they need not be used. 
    - The playlist name will be the theme of the playlist. Based on the playlist name, extract tracks in the data that matches
    the theme and respond with the list of tracks that should belong in the playlist.
    - The creativity parameter is similar to that of the creativity parameter for LLMs. It ranges from 0 to 1, where 1 is creative and 0 is not creative.
      This parameter tells the model how creative it should be when determining if a track belongs to the playlist theme. Naturally, higher creativity should
      mean more tracks are added to the playlist. 
    - The features in the data will be:
        Danceability: Measures how suitable a track is for dancing, based on rhythm and tempo. (Range: 0 to 1)
        Energy: Intensity and activity of a track. Higher values represent more energetic tracks. (Range: 0 to 1)
        Acousticness: Likelihood that a track is acoustic. (Range: 0 to 1)
        Instrumentalness: Measures the likelihood of no vocals. Higher means more instrumental. (Range: 0 to 1)
        Valence: Positiveness or happiness of a track. Higher values sound more positive. (Range: 0 to 1)
        Loudness: Average volume in decibels (dB). (Range: ~ -60 to 0 dB)
        Tempo: Speed of the track, measured in beats per minute (BPM). (Range: 0 to 300+ BPM)

    The response should only be the list of song separated by comma. i.e. songA, songB, songC, ... and so on. Note that songs needs to be related to the theme.
    For example, if there is no afro beats songs in the list of songs given, then no songs can be returned. In the case where no songs are returned, return an empty string.
    """
    prompt = f"""
    playlist name: {playlist_name}
    creativity: {creativity}
    data: {df.to_string(na_rep='NA')}
    """
    messages=[
        {"role": "system", "content": system_content},
        {"role": "user", "content": prompt},
    ]
    # print(prompt)
    print(f"Number of tokens number {len(encoding.encode(system_content)) + len(encoding.encode(prompt))}")
    print(f"Number of tokens: {count_tokens(messages)}")
    try:
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            creativity= 0.7
        )
    except openai.RateLimitError:
        raise Exception("Please try again in a few minutes. If this problem persists, contact ...")
        
    return response.choices[0].message.content
df = sp.get_df()
df = shuffle(df)
df = sp.get_df()
df = df.loc[5:150, ~df.columns.isin(['playlist', 'id'])]
start = time.monotonic()
string = generate_playlist_from_name('Morning drive', 0.5, df)
print(string)
print(f"Time taken: {round(time.monotonic()-start, 4)} seconds")

AttributeError: 'Spotify' object has no attribute 'get_df'

# Archive

In [ ]:
  prompt = f"""
      You are a spotify music playlist curator who organizes tracks into thematic playlists based on their audio features extracted
      from spotify api. 
       The inputs will be given in the following format:
        - playlist name: <str>
        - creativity: <int>
        - range of songs: <int> - <int>
        - special requests: <text>
        - data: ```<str>```

      Some important information on the inputs:
      - data: will be in csv format and the features below will be given for each track. These features may or may not be helpful 
        in clustering the tracks into playlists.
      - playlist name: will be the theme of the playlist. Based on the playlist name, only extract tracks in 'data' that matches
        the theme and respond with the list of tracks that should belong in the playlist.
      - creativity: is how creative you can be when curating the playlist. It ranges from 1 (all songs follow a particular musical feature related to the playlist name)
        to 10 (songs might have different musical features but are still somewhat related to the playlist name)
      - range of songs: determines the number of songs to be inputted, minimum - maximum value. The number of songs generated should be less than the maximum value.
        But, the number of songs need not be more than the minimum value.
      - Special requests: this is some additional information that will help you choose which songs to pick in the playlist.
      - The features in the data will be:
          name: Track name
          Danceability: Measures how suitable a track is for dancing, based on rhythm and tempo. (Range: 0 to 1)
          Energy: Intensity and activity of a track. Higher values represent more energetic tracks. (Range: 0 to 1)
          Acousticness: Likelihood that a track is acoustic. (Range: 0 to 1)
          Instrumentalness: Measures the likelihood of no vocals. Higher means more instrumental. (Range: 0 to 1)
          Valence: Positiveness or happiness of a track. Higher values sound more positive. (Range: 0 to 1)
          Loudness: Average volume in decibels (dB). (Range: ~ -60 to 0 dB)
          Tempo: Speed of the track, measured in beats per minute (BPM). (Range: 0 to 300+ BPM)
      - Remember to only extract songs included in 'data'
      Note that songs needs to be related to the theme. For example, if there is no afro beats songs in the list of songs given, then no songs can be returned. In the case where no songs are returned, return an empty string.
      Refer to the inputs below:
      playlist name: {playlist_name}
      creativity: {creativity}
      Range of songs: {min_tracks} - {max_tracks}
      Special requests: {special_request}
      data: ```{df.to_string(na_rep='NA')}```

      The response should only be the list of song separated by comma. i.e. songA, songB, songC, ... 
      """